In [ ]:
# Dùng các thư viện hỗ trợ như apiwho
# pandas

In [ ]:
# QUan trọng đầu tiên là lấy dữ liệu thô.
# Bài toán đặt ra:
# Đặc điểm về cách sống, đặc điểm vể di truyền, đặc điểm y tế môi trường đã và đang chi phối ra sao đến bệnh tim mạch? Qua đó, dựa vào dữ liệu lịch sử để phán đoán ra phần trăm chi phối của năm sau? 

In [ ]:
# Phân tích chi tiết
# Về cách sống (có hút thuốc hay không, chỉ số về cơ thể ra sao, có lười vận động hay không?, có uống rượu hay không?)
# Về di truyền (có bệnh tiểu đường không, có bệnh huyết áp không, có cao máu không)
# Về môi trường (môi trường có ô nhiễm không, cơ sở y tế ra sao?)

# --> Như vậy, mình phân tích bệnh tim mạch qua 3 nhóm yếu tố, tổng cộng 9 yếu tố nguy cơ
# --> Mình cần có 9 tập dữ liệu khác nhau? --> Sau đó tiến hành gom nhóm theo quốc gia (lấy ở Việt Nam), và thời gian.

In [ ]:
# Định dạng lưu trữ dữ liệu thô --> file raw.csv

# Chương trình cào dữ liệu

In [1]:
import requests
import json
import pandas as pd

In [9]:
class whoAmI:
    def __init__(self):
        self.getDim='https://ghoapi.azureedge.net/api/Dimension'
        self.getDimValues='https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues'
        self.getIndValues='https://ghoapi.azureedge.net/api/Indicator'
        self.getIndFilterVal='https://ghoapi.azureedge.net/api/Indicator?$filter=contains(IndicatorName,'
        self.getIndFilterSpecVal='https://ghoapi.azureedge.net/api/Indicator?$filter=IndicatorName eq '
        self.getIndData='https://ghoapi.azureedge.net/api/'

In [7]:
who_urls = whoAmI()

In [12]:
def request_content(path):
    resp = requests.get(path)
    data = json.loads(resp.content)
    return data['value']


In [40]:
def request_country_data(obj):
    params = {"$filter": "ParentLocationCode eq 'WPRO'"}
    resp = requests.get(who_urls.getIndData + '/' + obj)
    data = json.loads(resp.content)
    return data['value']

In [ ]:
# Lấy các chiều dữ liệu được thu thập
# print(request_content(who_urls.getDim))

[{'Code': 'ADVERTISINGTYPE', 'Title': 'SUBSTANCE_ABUSE_ADVERTISING_TYPES'}, {'Code': 'AGEGROUP', 'Title': 'Age Group'}, {'Code': 'ALCOHOLTYPE', 'Title': 'Beverage Types'}, {'Code': 'AMRGLASSCATEGORY', 'Title': 'AMR GLASS Category'}, {'Code': 'ANTIBIOTIC', 'Title': 'Antibiotic'}, {'Code': 'ARCHIVE', 'Title': 'Archive date'}, {'Code': 'ASSISTIVETECHBARRIER', 'Title': 'Barriers to accessing assistive products'}, {'Code': 'ASSISTIVETECHFUNDING', 'Title': 'Funding for assistive tech products'}, {'Code': 'ASSISTIVETECHPRODUCT', 'Title': 'Assistive technology product'}, {'Code': 'ASSISTIVETECHSATIACTIVITY', 'Title': 'Satisfaction with assistive products for different environments and activities'}, {'Code': 'ASSISTIVETECHSATISERVICE', 'Title': 'Satisfaction with assistive products and related services'}, {'Code': 'ASSISTIVETECHSOURCE', 'Title': 'Sources of assistive products'}, {'Code': 'ASSISTIVETECHSUBQUESTION', 'Title': 'Assistive technology subquestion'}, {'Code': 'ASSISTIVETECHTRAVELDISTA

In [18]:
# Lấy các Indicator
indicators = pd.DataFrame(request_content(who_urls.getDimValues))

In [19]:
print(indicators)

    Code                                              Title Dimension  \
0    ABW                                              Aruba   COUNTRY   
1    AFG                                        Afghanistan   COUNTRY   
2    AGO                                             Angola   COUNTRY   
3    AIA                                           Anguilla   COUNTRY   
4    ALB                                            Albania   COUNTRY   
..   ...                                                ...       ...   
229  XKX  Kosovo (in accordance with UN Security Council...   COUNTRY   
230  YEM                                              Yemen   COUNTRY   
231  ZAF                                       South Africa   COUNTRY   
232  ZMB                                             Zambia   COUNTRY   
233  ZWE                                           Zimbabwe   COUNTRY   

    ParentDimension ParentCode            ParentTitle  
0            REGION        AMR               Americas  
1          

In [ ]:
# Mã bộ dữ liệu của dự án
smoke = 'Adult_curr_tob_smoking' # Hút thuốc lá
bmi1 = 'NCD_BMI_30C'
bmi2 = 'NCD_BMI_25C' # 
lazy = 'NCD_PAA' # Lười vận động
alcohol = 'SA_0000000001' # Cồn
diabetes = 'NCD_GLUC_03' # Tiểu đường
hypertension = 'NCD_HYP_PREVALENCE_A' # Huyết áp
cholesterol = 'NCD_CHOL_3' # Mỡ máu
pollution = 'AIR_11' # Ô nhiễm 
available = 'UHC_INDEX_REPORTED' # Tiếp cận cơ sở y tế
target = 'WHOSIS_000001' # Bệnh tim mạch


In [ ]:
# Các mục tiêu dữ liệu cần lấy
objectives = [smoke, bmi1, bmi2, diabetes, hypertension, cholesterol, pollution, available, target]

In [48]:
def storage_data(objective):
    raw_data = request_country_data(objective)
    fields = ['ParentLocationCode', 'Dim1', 'Value', 'NumericValue', 'Date', 'IndicatorCode']

    records = []
    for row in raw_data:
        _tempo = dict()
        for field in fields:
            _tempo[field] = row[field]
        records.append(_tempo)

    storage = "raw/" + objective + '.json'
    with open(storage, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

In [53]:
print("Tiến hành lưu trữ dữ liệu.....")
for objective in objectives:
    print("Xử lý:", objective)
    storage_data(objective)
    print("Hoàn thành xử lý:", objective)

Tiến hành lưu trữ dữ liệu.....
Xử lý: Adult_curr_tob_smoking
Hoàn thành xử lý: Adult_curr_tob_smoking
Xử lý: NCD_BMI_30C
Hoàn thành xử lý: NCD_BMI_30C
Xử lý: NCD_BMI_25C
Hoàn thành xử lý: NCD_BMI_25C
Xử lý: NCD_GLUC_03
Hoàn thành xử lý: NCD_GLUC_03
Xử lý: NCD_HYP_PREVALENCE_A
Hoàn thành xử lý: NCD_HYP_PREVALENCE_A
Xử lý: AP_MAX_PM25


JSONDecodeError: Expecting value: line 1 column 1 (char 0)